# C4 — Auto-Research Editing Loop (Colab, A100)

Runs the society-vs-blind-vs-static critic ablation inside a 10-step
auto-refinement editing loop, then renders the deliverable figures.

**Requirements:** an **A100 40GB** runtime (Runtime → Change runtime type → A100),
and an HF token whose account has **accepted the `black-forest-labs/FLUX.1-Kontext-dev`
license** (it is a gated model). On smaller GPUs pass `--cpu-offload`, or
`--editor instructpix2pix` (ungated) as a fallback.

Pipeline: `script/c4_refine.py` (run) → `scripts/analysis/c4_trajectory.py` +
`c4_qualitative.py`. All outputs go under a single `--output-root`
(`edits/`, `logs/`, `analysis/c4.json`, `analysis/figs/`).

## 1. Clone the repo + install the GPU stack

In [ ]:
# Clone (replace with your remote if different)
![ -d SyntheticAudience ] || git clone https://github.com/<your-org>/SyntheticAudience.git
%cd SyntheticAudience
!pip -q install -r requirements-colab.txt

## 2. Hugging Face token (FLUX.1-Kontext-dev is gated)

Sets `HF_TOKEN` in the **kernel environment** so both `!python` cells and
`huggingface_hub` see it.

- **Do not paste the token into a cell** — anything typed in a cell is saved in
  the notebook. Use Colab **Secrets** (🔑 icon → add `HF_TOKEN`, enable for this
  notebook), or the hidden `getpass` prompt below.
- `!export HF_TOKEN=...` does **not** work: each `!` line is a separate subshell,
  so the variable never reaches later cells.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # Colab Secrets (preferred)
except Exception:
    import getpass
    os.environ['HF_TOKEN'] = getpass.getpass('HF token (input hidden): ')
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])  # gated FLUX.1-Kontext-dev + private data repos

## 2.5 Set the output folder (write straight to Drive)

Every script takes `--output-root`. Point it at a named Drive folder and all
edited images (`edits/`), per-step logs (`logs/`), and deliverables
(`analysis/c4.json` + `analysis/figs/`) are written there directly — no symlinks,
and the run is `--resume`-able across sessions because the logs + edit cache
persist. Give each run a distinct name (e.g. `.../c4_run1`).

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')  # idempotent if already mounted

# Named Drive folder for this work. Give each experiment a distinct run name.
OUTPUT_ROOT = '/content/drive/MyDrive/SyntheticAudience_C4/c4_run1'
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print('C4 outputs ->', OUTPUT_ROOT, '(edits/, logs/, analysis/)')

## 3. Fetch data (EVA + PARA)
`data/` is git-ignored and pulled from your private HF datasets via the repo's
helper. Source images live on local Colab disk; only the C4 *outputs* go to
`OUTPUT_ROOT` on Drive. (To avoid re-downloading each session, see Section 8B.)

In [ ]:
# HF_TOKEN was set in the kernel env in section 2, so !python inherits it here.
!python scripts/fetch_from_hf.py eva para

## 4. GPU check

In [ ]:
import torch
p = torch.cuda.get_device_properties(0)
gb = p.total_memory / 1e9
print(f'{p.name}  {gb:.1f} GB')
if gb < 38:
    print('WARNING: <40GB — add --cpu-offload (slow) or use --editor instructpix2pix.')

## 5. Smoke test (2 images, 3 steps) — confirm the whole loop runs

In [ ]:
!python script/c4_refine.py --dataset eva --n-images 2 \
    --conditions static,blind,society --editor flux --steps 3 --candidates 2 \
    --output-root "{OUTPUT_ROOT}"

## 6. Full run
~150 images per dataset × 4 conditions × 10 steps × K=3. Long (overnight-ish);
cached + resumable (`--resume`), shardable across GPUs (`--shard i/N`).

In [ ]:
!python script/c4_refine.py --dataset both --n-images 150 \
    --conditions static,blind,society,reward_only \
    --editor flux --steps 10 --candidates 3 --resume \
    --output-root "{OUTPUT_ROOT}"

## 7. Deliverables — trajectory curves, AUC, drift, diversity, qualitative grid

In [ ]:
%cd scripts/analysis
!python c4_trajectory.py --output-root "{OUTPUT_ROOT}"
!python c4_qualitative.py --output-root "{OUTPUT_ROOT}"
%cd ../..

In [ ]:
from IPython.display import Image as IImage, display
for f in ['c4_trajectory.png', 'c4_drift.png', 'c4_diversity.png', 'c4_qualitative.png']:
    display(IImage(filename=f'{OUTPUT_ROOT}/analysis/figs/{f}'))

## 8. Speed / persistence tips

**A) If writing directly to Drive is slow** (FUSE is sluggish for the ~20k+ tiny
PNGs at full scale): set `OUTPUT_ROOT` to a **local** path (e.g. `/content/c4_run1`)
so the run is fast, then rsync to Drive periodically / after each shard — safe to
re-run, only changed files copy.

**B) Persist the fetched source data** to skip re-downloading each session:
symlink `data/eva` and `data/para` to Drive *before* the fetch cell.

In [ ]:
# A) local run -> rsync to Drive (only if OUTPUT_ROOT is a LOCAL path)
LOCAL_ROOT = '/content/c4_run1'
DRIVE_DEST = '/content/drive/MyDrive/SyntheticAudience_C4/c4_run1'
!mkdir -p "{DRIVE_DEST}"
!rsync -a --info=progress2 "{LOCAL_ROOT}/" "{DRIVE_DEST}/"

# B) persist source data too (run BEFORE the fetch cell):
# import os; from pathlib import Path
# for ds in ('eva', 'para'):
#     tgt = f'/content/drive/MyDrive/SyntheticAudience_C4/data_{ds}'
#     Path(tgt).mkdir(parents=True, exist_ok=True)
#     lp = Path('/content/SyntheticAudience/data')/ds
#     if not lp.exists(): lp.symlink_to(tgt)